# WP model demo

Thin demo: reads `plays_scored.parquet` (produced by `ffep score`) and plots the win-probability curve for one game. No model fitting happens here -- run `ffep train --model wp` then `ffep score` to (re)produce the scored data.

**Synthetic clock warning:** `half_seconds_remaining` used by the WP model is **synthetic** (`1200 / max(play_id)` per half), not a real game clock -- real Hudl clock data has not been delivered (REQ-S1-02 pending). The x-axis below is play sequence, not wall-clock time; do not read this chart as time-calibrated. See `docs/pipeline.md`'s Known Limitations section.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

scored = pl.read_parquet(Path('../data/processed') / 'plays_scored.parquet')

# Pick one game with a full-length, non-null WP curve.
candidate = (
    scored.filter(pl.col('wp').is_not_null())
    .group_by('game_id')
    .len()
    .sort('len', descending=True)
    .row(0)
)
game_id = candidate[0]
game = scored.filter(pl.col('game_id') == game_id).sort(['half', 'play_id'])
home_team, away_team = game['home_team'][0], game['away_team'][0]
print(f'game_id={game_id}  {home_team} (home) vs {away_team} (away)  n_plays={game.height}')

## Win-probability curve (play sequence, not real time)

In [ ]:
x = range(game.height)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, game['home_wp'], label=f'{home_team} (home) WP')
ax.axhline(0.5, color='grey', linewidth=0.8, linestyle='--')
ax.set_xlabel('play sequence (SYNTHETIC clock -- not wall-clock time, REQ-S1-02)')
ax.set_ylabel('home win probability')
ax.set_ylim(0, 1)
ax.set_title(f'WP curve: {home_team} vs {away_team} ({game_id})')
ax.legend()
plt.show()